#  Análise de Desempenho — Lojas Físicas

**Squad 3 — Batch Lojas Físicas | Luiz Henrique Portácio**

Este notebook apresenta os indicadores de desempenho das lojas físicas
para consumo pelos gestores da operação. Os dados passam por um
pipeline de qualidade (Bronze → Silver → Gold) antes de chegarem aqui.

> **Nota:** Este notebook é uma camada de validação e análise exploratória.
> O dashboard oficial de BI será disponibilizado via **Looker**, conectado
> diretamente ao SQL Server (`squad3.gold_physical_lojas`).

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = get_adls_options()

In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, count, when,
    round as spark_round, avg as spark_avg
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go

# ── Paleta de cores ──────────────────────────────────────────────────────────
# Cores com contraste garantido (WCAG AA) — sem branco sobre branco
COR_POSITIVO = "#1B5E20"   # verde escuro — legível em fundo branco
COR_NEGATIVO = "#B71C1C"   # vermelho escuro — legível em fundo branco
COR_ALERTA   = "#E65100"   # laranja escuro — legível em fundo branco
COR_AZUL     = "#0D47A1"   # azul escuro — legível em fundo branco
COR_CINZA    = "#424242"   # cinza escuro — legível em fundo branco
COR_FUNDO    = "#ECEFF1"   # fundo cinza claro — neutro

# Sequência de cores para múltiplas lojas (todas com contraste sobre branco)
PALETA_DISCRETA = [
    "#1565C0", "#2E7D32", "#6A1B9A", "#E65100",
    "#00695C", "#AD1457", "#4527A0", "#37474F",
]

# Configuração global matplotlib
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#FAFAFA",
    "axes.edgecolor":   "#BDBDBD",
    "axes.labelcolor":  "#212121",
    "text.color":       "#212121",
    "xtick.color":      "#424242",
    "ytick.color":      "#424242",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.size":        10,
})

# Configuração global Plotly
LAYOUT_BASE = dict(
    plot_bgcolor="white",
    paper_bgcolor="white",
    hoverlabel=dict(bgcolor="#212121", font_color="white", font_size=12),
)
def formatar_reais(valor):
    """Formata valor numérico como string BRL com separadores."""
    if valor is None or valor != valor:
        return "—"
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def cor_barra(valor, limiar=0):
    """Retorna cor com contraste garantido conforme o valor vs limiar."""
    return COR_POSITIVO if valor >= limiar else COR_NEGATIVO

---
## Sumário Executivo

In [0]:
df_gold = read_delta(GOLD_LOJAS_PATH, adls_options)

anos       = sorted([r.ano for r in df_gold.select("ano").distinct().collect()])
ano_ref    = max(anos) if anos else None
ano_ini    = min(anos) if anos else None
periodo_hist = f"{ano_ini}\u2013{ano_ref}" if (ano_ini and ano_ref and ano_ini != ano_ref) else str(ano_ref)
df_ref     = df_gold.filter(col("ano") == ano_ref)

total_lojas      = df_gold.select("id_loja").distinct().count()
receita_total    = df_ref.agg(spark_sum("receita_mes")).collect()[0][0] or 0
transacoes_total = df_ref.agg(spark_sum("qtd_transacoes")).collect()[0][0] or 0
ticket_medio_med = df_ref.filter(col("ticket_medio").isNotNull()).agg(
    spark_avg("ticket_medio")).collect()[0][0] or 0
yoy_medio        = df_gold.filter(col("crescimento_yoy_pct").isNotNull()).agg(
    spark_avg("crescimento_yoy_pct")).collect()[0][0] or 0

receita_fmt    = formatar_reais(receita_total)
ticket_fmt     = formatar_reais(ticket_medio_med)
yoy_sinal   = "▲" if yoy_medio >= 0 else "▼"

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║     SUMÁRIO EXECUTIVO — LOJAS FÍSICAS                            ║
║     Período histórico : {periodo_hist:<41}║
║     Ano de referência : {str(ano_ref):<41}║
╠══════════════════════════════════════════════════════════════════╣
║  🏪  Total de lojas ativas    :  {total_lojas:<30}║
║  💰  Receita total ({ano_ref})  :  {receita_fmt:<30}║
║  🛒  Total de transações      :  {int(transacoes_total):>15,}               ║
║  🎯  Ticket médio (rede)      :  {ticket_fmt:<30}║
║  📈  Crescimento YoY médio    :  {yoy_sinal} {yoy_medio:+.1f}%                          ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
##  Qualidade dos Dados

Integridade verificada automaticamente pelo pipeline antes de cada carga.
Registros com problemas são **isolados** e não afetam os KPIs abaixo.

In [0]:
df_dq = (
    read_delta(SILVER_DQ_METRICS_PATH, adls_options)
    .filter(col("tabela") == "physical_lojas")
    .groupBy("regra")
    .agg(
        spark_sum("qtd_afetados").alias("qtd_afetados"),
        spark_sum("qtd_total").alias("qtd_total"),
    )
    .orderBy("regra")
)
pdf_dq = df_dq.toPandas()

NOMES_REGRA = {
    "01_pk_id_loja_nula_ou_duplicada": "Integridade do ID da Loja",
    "02_cnpj_invalido":                "Validade do CNPJ",
    "03_estado_loja_uf_invalido":      "Validade do Estado (UF)",
}
pdf_dq["regra_negocio"] = pdf_dq["regra"].map(lambda x: NOMES_REGRA.get(x, x))
pdf_dq["pct_valido"]    = (
    100 - pdf_dq["qtd_afetados"] / pdf_dq["qtd_total"] * 100
).round(1)

# ── Score cards minimalistas ─────────────────────────────────────────────────
# Design: fundo branco, linha colorida no topo, tipografia limpa.
# Sem emoji misturado ao número, sem título colorido fora do card.
n   = len(pdf_dq)
fig = plt.figure(figsize=(4.2 * n, 3.2), facecolor="white")

for idx, (_, row) in enumerate(pdf_dq.iterrows()):
    pct = row["pct_valido"]
    erros = int(row["qtd_afetados"])

    if pct >= 95:
        cor_linha, label_status, cor_label = "#1B5E20", "✓ Aprovado", "#1B5E20"
    elif pct >= 80:
        cor_linha, label_status, cor_label = "#E65100", "⚠ Atenção",  "#E65100"
    else:
        cor_linha, label_status, cor_label = "#B71C1C", "✗ Crítico",  "#B71C1C"

    ax = fig.add_subplot(1, n, idx + 1)
    ax.set_facecolor("white")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.axis("off")

    # Linha colorida no topo do card
    # ax.plot com transAxes — axhline não aceita transform como argumento
    ax.plot([0.08, 0.92], [0.97, 0.97], color=cor_linha,
            linewidth=5, solid_capstyle="butt",
            transform=ax.transAxes, clip_on=False)

    # Percentual — destaque máximo, cor do status
    ax.text(0.5, 0.68, f"{pct:.1f}%",
            ha="center", va="center",
            fontsize=38, fontweight="bold", color=cor_linha,
            transform=ax.transAxes)

    # Status — pequeno, alinhado abaixo do número
    ax.text(0.5, 0.50, label_status,
            ha="center", va="center",
            fontsize=9, fontweight="bold", color=cor_label,
            transform=ax.transAxes)

    # Separador sutil
    ax.plot([0.10, 0.90], [0.42, 0.42], color="#E0E0E0",
            linewidth=0.8, transform=ax.transAxes, clip_on=False)

    # Nome da regra — cinza escuro, legível
    ax.text(0.5, 0.30, row["regra_negocio"],
            ha="center", va="center",
            fontsize=9, color="#424242", fontweight="bold",
            transform=ax.transAxes)

    # Detalhe de erros — cinza claro, discreto
    detalhe = f"{erros:,} erro(s)" if erros > 0 else "Sem erros"
    ax.text(0.5, 0.12, detalhe,
            ha="center", va="center",
            fontsize=8, color="#757575",
            transform=ax.transAxes)

    # Borda externa sutil ao redor do card
    for side in ["top", "bottom", "left", "right"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#E0E0E0")
        ax.spines[side].set_linewidth(0.8)

fig.suptitle("Qualidade dos Dados — Lojas Físicas",
             fontsize=12, fontweight="bold", color="#212121", y=1.04)
plt.tight_layout(pad=1.2)
plt.show()

In [0]:
# ── Gráfico interativo — % válido por critério ────────────────────────────────
cores_barra = [COR_POSITIVO if p >= 95 else (COR_ALERTA if p >= 80 else COR_NEGATIVO)
               for p in pdf_dq["pct_valido"]]

fig = go.Figure(go.Bar(
    x=pdf_dq["regra_negocio"],
    y=pdf_dq["pct_valido"],
    marker_color=cores_barra,
    # Texto DENTRO das barras (nunca "outside" — evita sobreposição e
    # texto sobre fundo branco)
    text=pdf_dq["pct_valido"].apply(lambda x: f"{x:.1f}%"),
    textposition="inside",
    textfont=dict(color="white", size=13, family="Arial Black"),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Registros válidos: <b>%{y:.1f}%</b><br>"
        "Com problema: <b>%{customdata:,}</b><extra></extra>"
    ),
    customdata=pdf_dq["qtd_afetados"],
    width=0.5,
))
fig.add_hline(y=95, line_dash="dot", line_color=COR_CINZA, line_width=1.5,
              annotation_text="Meta mínima: 95%",
              annotation_font_color=COR_CINZA,
              annotation_position="top right")
fig.update_layout(
    **LAYOUT_BASE,
    title=dict(text="% de Registros Válidos por Critério de Qualidade",
               font=dict(size=14, color="#212121")),
    yaxis=dict(title="% Válido", range=[0, 108], ticksuffix="%",
               gridcolor="#E0E0E0", tickfont=dict(color="#424242")),
    xaxis=dict(title="Critério de Qualidade",
               tickfont=dict(color="#424242")),
    height=400,
    margin=dict(t=70, b=60, l=80, r=120),
)
fig.show()

---
## KPI 4 — Receita Total por Loja
*Quanto cada loja contribui para a receita total da rede.*

In [0]:
df_receita = (
    df_gold
    .groupBy("nome_loja", "ano")
    .agg(spark_round(spark_sum("receita_mes"), 2).alias("receita_total"))
    .orderBy("ano", col("receita_total").desc())
)
pdf_receita = df_receita.toPandas()

if not pdf_receita.empty:
    pdf_ref = pdf_receita[pdf_receita["ano"] == ano_ref].copy()
    pdf_ref = pdf_ref.sort_values("receita_total", ascending=True)
    total_rede = pdf_ref["receita_total"].sum()
    pdf_ref["participacao"] = (pdf_ref["receita_total"] / total_rede * 100).round(1)

    top = pdf_ref.iloc[-1] if not pdf_ref.empty else None
    bot = pdf_ref.iloc[0]  if not pdf_ref.empty else None
    top3_pct = pdf_ref.nlargest(3, "receita_total")["participacao"].sum() if not pdf_ref.empty else 0
    if top is not None:
        print(f"💡 Top 3 lojas respondem por {top3_pct:.1f}% da receita total da rede.")
        print(f"   🥇 Maior: {top['nome_loja']} — {formatar_reais(top['receita_total'])} ({top['participacao']:.1f}%)")
        print(f"   🔻 Menor: {bot['nome_loja']} — {formatar_reais(bot['receita_total'])} ({bot['participacao']:.1f}%)")

    # ── Plotly — barras horizontais com cor única e contraste garantido ──────
    # Cor sólida escura com texto dentro (evita branco sobre branco)
    fig = go.Figure(go.Bar(
        y=pdf_ref["nome_loja"],
        x=pdf_ref["receita_total"],
        orientation="h",
        marker_color=COR_AZUL,
        marker_line=dict(color="#0A2472", width=0.5),
        text=pdf_ref.apply(
            lambda r: f"  {formatar_reais(r['receita_total'])}  ({r['participacao']:.1f}%)",
            axis=1
        ),
        textposition="inside",
        # Texto escuro — nunca "white" em textposition="inside" com escala de cor
        textfont=dict(color="white", size=10),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Receita: <b>%{x:,.2f}</b><br>"
            "Participação: <b>%{customdata:.1f}%</b><extra></extra>"
        ),
        customdata=pdf_ref["participacao"],
        width=0.65,
    ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text=f"Receita Total por Loja — {ano_ref}",
                   font=dict(size=14, color="#212121")),
        xaxis=dict(title="Receita (R$)", tickprefix="R$ ",
                   tickformat=",.0f", gridcolor="#E0E0E0",
                   tickfont=dict(color="#424242")),
        yaxis=dict(title="", tickfont=dict(color="#212121", size=10)),
        height=max(380, len(pdf_ref) * 42),
        margin=dict(t=70, b=60, l=160, r=80),
    )
    fig.show()

    # ── Matplotlib (estático para PR) ────────────────────────────────────────
    if len(pdf_receita["ano"].unique()) > 1:
        pdf_pivot = pdf_receita.pivot(index="nome_loja", columns="ano",
                                       values="receita_total").fillna(0)
        n_anos    = len(pdf_pivot.columns)
        cores_mat = PALETA_DISCRETA[:n_anos]
        fig, ax   = plt.subplots(figsize=(10, max(4, len(pdf_pivot) * 0.55)))
        pdf_pivot.plot(kind="barh", ax=ax, color=cores_mat, edgecolor="white",
                       linewidth=0.5)
        ax.set_title(f"Receita Total por Loja — {periodo_hist}", fontsize=12,
                     fontweight="bold", color="#212121")
        ax.set_xlabel("Receita (R$)", color="#424242")
        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"R$ {x:,.0f}")
        )
        ax.legend(title="Ano", fontsize=8, title_fontsize=9)
        ax.tick_params(colors="#424242")
        plt.tight_layout()
        plt.show()

---
##  KPI 5 — Crescimento Anual (YoY)
*Variação percentual da receita vs o ano anterior. ▲ Verde = crescimento | ▼ Vermelho = queda.*

In [0]:
df_yoy = (
    df_gold
    .select("nome_loja", "ano", "crescimento_yoy_pct")
    .distinct()
    .filter(col("crescimento_yoy_pct").isNotNull())
    .orderBy("nome_loja", "ano")
)
pdf_yoy = df_yoy.toPandas()

if not pdf_yoy.empty:
    pdf_yoy_ref = pdf_yoy[pdf_yoy["ano"] == ano_ref].copy()
    pdf_yoy_ref = pdf_yoy_ref.sort_values("crescimento_yoy_pct")

    n_positivo = (pdf_yoy_ref["crescimento_yoy_pct"] > 0).sum()
    n_negativo = (pdf_yoy_ref["crescimento_yoy_pct"] <= 0).sum()
    print(f"📊 {ano_ref}: {n_positivo} loja(s) crescendo 🟢 | {n_negativo} loja(s) em queda 🔴")

    # ── Plotly — bullet chart divergente ─────────────────────────────────────
    # Cor determinada pelo valor (positivo/negativo), texto sempre fora
    # mas com margem suficiente para não sair do gráfico
    cores_yoy = [COR_POSITIVO if v >= 0 else COR_NEGATIVO
                 for v in pdf_yoy_ref["crescimento_yoy_pct"]]

    # Calcula range para garantir espaço para os labels fora das barras
    val_max = pdf_yoy_ref["crescimento_yoy_pct"].abs().max()
    x_range = [-val_max * 1.35, val_max * 1.35]

    fig = go.Figure(go.Bar(
        y=pdf_yoy_ref["nome_loja"],
        x=pdf_yoy_ref["crescimento_yoy_pct"],
        orientation="h",
        marker_color=cores_yoy,
        marker_line=dict(color="white", width=0.5),
        # Texto FORA — cor escura combinando com a cor da barra (não branco)
        text=pdf_yoy_ref["crescimento_yoy_pct"].apply(lambda x: f"{x:+.1f}%"),
        textposition="outside",
        textfont=dict(
            color=[COR_POSITIVO if v >= 0 else COR_NEGATIVO
                   for v in pdf_yoy_ref["crescimento_yoy_pct"]],
            size=11,
            family="Arial Black",
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "YoY: <b>%{x:+.1f}%</b><extra></extra>"
        ),
        width=0.6,
    ))
    fig.add_vline(x=0, line_color="#212121", line_width=1.5)
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text=f"Crescimento YoY por Loja — {ano_ref} vs {ano_ref - 1 if ano_ref else '?'}",
            font=dict(size=14, color="#212121"),
        ),
        xaxis=dict(
            title="Variação (%)", ticksuffix="%",
            range=x_range,
            gridcolor="#E0E0E0",
            zeroline=True, zerolinecolor="#212121", zerolinewidth=1.5,
            tickfont=dict(color="#424242"),
        ),
        yaxis=dict(title="", tickfont=dict(color="#212121", size=10)),
        height=max(380, len(pdf_yoy_ref) * 42),
        margin=dict(t=70, b=60, l=160, r=100),
    )
    fig.show()

    # ── Matplotlib (estático) ────────────────────────────────────────────────
    if len(pdf_yoy["ano"].unique()) > 1:
        fig, ax = plt.subplots(figsize=(9, 5))
        anos_unicos = sorted(pdf_yoy["ano"].unique())
        for nome, grupo in pdf_yoy.groupby("nome_loja"):
            ultimo_yoy = grupo.iloc[-1]["crescimento_yoy_pct"]
            cor = COR_POSITIVO if ultimo_yoy > 0 else COR_NEGATIVO
            ax.plot(grupo["ano"], grupo["crescimento_yoy_pct"],
                    marker="o", label=nome, color=cor, linewidth=1.8,
                    markersize=5)
        ax.axhline(0, color="#212121", linewidth=1.2, linestyle="--", zorder=5)
        # fill_between com limites FIXOS (não get_xlim que é instável)
        x_min, x_max = min(anos_unicos) - 0.3, max(anos_unicos) + 0.3
        ax.set_xlim(x_min, x_max)
        ymin, ymax = ax.get_ylim()
        if ymax > 0:
            ax.axhspan(0, ymax, alpha=0.04, color=COR_POSITIVO, zorder=0)
        if ymin < 0:
            ax.axhspan(ymin, 0, alpha=0.04, color=COR_NEGATIVO, zorder=0)
        ax.set_title("Crescimento YoY por Loja", fontsize=12,
                     fontweight="bold", color="#212121")
        ax.set_ylabel("Variação (%)", color="#424242")
        ax.set_xlabel("Ano", color="#424242")
        # Legenda com no máximo 3 colunas para não sobrepor
        n_lojas = pdf_yoy["nome_loja"].nunique()
        ax.legend(fontsize=7, ncol=min(3, max(1, n_lojas // 3)),
                  loc="upper left", framealpha=0.9)
        plt.tight_layout()
        plt.show()
else:
    print("⚠️  YoY requer pelo menos 2 anos de histórico por loja.")

---
## KPI 6 — Volume de Transações por Mês
*Frequência de compras ao longo do tempo — revela sazonalidade e tendências.*

In [0]:
df_trans = (
    df_gold
    .select("nome_loja", "ano", "mes", "qtd_transacoes")
    .distinct()
    .orderBy("nome_loja", "ano", "mes")
)
pdf_trans = df_trans.toPandas()

if not pdf_trans.empty:
    pdf_trans["ano_mes"] = (
        pdf_trans["ano"].astype(str) + "-" +
        pdf_trans["mes"].astype(str).str.zfill(2)
    )
    pdf_trans = pdf_trans.sort_values("ano_mes")

    MESES = {1:"Jan",2:"Fev",3:"Mar",4:"Abr",5:"Mai",6:"Jun",
             7:"Jul",8:"Ago",9:"Set",10:"Out",11:"Nov",12:"Dez"}

    # Sazonalidade
    pdf_sazon = pdf_trans.groupby("mes")["qtd_transacoes"].mean().reset_index()
    mes_pico  = pdf_sazon.loc[pdf_sazon["qtd_transacoes"].idxmax(), "mes"]
    mes_baixo = pdf_sazon.loc[pdf_sazon["qtd_transacoes"].idxmin(), "mes"]
    print(f"📅 Mês com MAIS movimento  : {MESES[mes_pico]}")
    print(f"📅 Mês com MENOS movimento : {MESES[mes_baixo]}")

    # ── Plotly — série temporal ───────────────────────────────────────────────
    fig = go.Figure()
    for i, (nome, grupo) in enumerate(pdf_trans.groupby("nome_loja")):
        fig.add_trace(go.Scatter(
            x=grupo["ano_mes"],
            y=grupo["qtd_transacoes"],
            name=nome,
            mode="lines+markers",
            line=dict(color=PALETA_DISCRETA[i % len(PALETA_DISCRETA)], width=2),
            marker=dict(size=5),
            hovertemplate=f"<b>{nome}</b><br>Mês: %{{x}}<br>Transações: <b>%{{y:,}}</b><extra></extra>",
        ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="Transações por Loja por Mês",
                   font=dict(size=14, color="#212121")),
        xaxis=dict(title="Mês/Ano", tickangle=-45,
                   tickfont=dict(color="#424242", size=9),
                   gridcolor="#E0E0E0"),
        yaxis=dict(title="Qtd. Transações", gridcolor="#E0E0E0",
                   tickfont=dict(color="#424242")),
        height=460,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02,
                    xanchor="left", x=0, font=dict(size=9)),
        margin=dict(t=100, b=80, l=80, r=40),
    )
    fig.show()

    # ── Plotly — sazonalidade (barras com cor contrastante) ──────────────────
    cores_sazon = [COR_POSITIVO if m == mes_pico else
                   (COR_NEGATIVO if m == mes_baixo else COR_AZUL)
                   for m in pdf_sazon["mes"]]

    fig2 = go.Figure(go.Bar(
        x=[MESES[m] for m in pdf_sazon["mes"]],
        y=pdf_sazon["qtd_transacoes"],
        marker_color=cores_sazon,
        marker_line=dict(color="white", width=0.5),
        text=pdf_sazon["qtd_transacoes"].apply(lambda x: f"{x:,.0f}"),
        textposition="inside",
        textfont=dict(color="white", size=11),
        hovertemplate="<b>%{x}</b><br>Média: <b>%{y:,.0f}</b> transações<extra></extra>",
        width=0.65,
    ))
    fig2.update_layout(
        **LAYOUT_BASE,
        title=dict(text="Sazonalidade — Média de Transações por Mês (Toda a Rede)",
                   font=dict(size=14, color="#212121")),
        xaxis=dict(title="Mês", tickfont=dict(color="#424242")),
        yaxis=dict(title="Média de Transações", gridcolor="#E0E0E0",
                   tickfont=dict(color="#424242")),
        height=380,
        margin=dict(t=70, b=60, l=80, r=40),
    )
    fig2.show()

    # ── Matplotlib (estático) ────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (nome, grupo) in enumerate(pdf_trans.groupby("nome_loja")):
        cor = PALETA_DISCRETA[i % len(PALETA_DISCRETA)]
        g = grupo.sort_values("ano_mes")
        ax.plot(g["ano_mes"], g["qtd_transacoes"],
                marker="o", markersize=3, label=nome,
                color=cor, linewidth=1.5)
    ax.set_title("Transações por Loja por Mês", fontsize=12,
                 fontweight="bold", color="#212121")
    ax.set_xlabel("Mês/Ano", color="#424242")
    ax.set_ylabel("Transações", color="#424242")
    # Rotação com ha="right" para evitar sobreposição
    ax.set_xticks(ax.get_xticks())
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    n_lojas = pdf_trans["nome_loja"].nunique()
    ax.legend(fontsize=7, ncol=min(4, max(1, n_lojas // 2)),
              loc="upper left", framealpha=0.9)
    plt.tight_layout()
    plt.show()

---
##  KPI 7 — Contexto Demográfico (IBGE 2022)
*Relaciona cada loja com a população do município onde está instalada.*

In [0]:
df_pop = (
    df_gold
    .select("nome_loja", "cidade_loja", "estado_tratado",
            "populacao_cidade", "flag_sem_match_ibge")
    .distinct()
    .orderBy(col("populacao_cidade").desc())
)
pdf_pop = df_pop.toPandas()

total    = len(pdf_pop)
c_match  = (~pdf_pop["flag_sem_match_ibge"]).sum()
pct_ok   = round(c_match / total * 100, 1) if total else 0
pdf_com  = pdf_pop[pdf_pop["populacao_cidade"].notna()].copy()
pdf_com["populacao_int"] = pdf_com["populacao_cidade"].astype(int)

# ── Score cards minimalistas — mesmo padrão da célula 8 ─────────────────────
fig = plt.figure(figsize=(8.4, 3.2), facecolor="white")

# Dados dos dois cards
cor_cob  = "#1B5E20" if pct_ok >= 90 else ("#E65100" if pct_ok >= 70 else "#B71C1C")
lbl_cob  = "✓ Alta cobertura" if pct_ok >= 90 else ("⚠ Cobertura parcial" if pct_ok >= 70 else "✗ Baixa cobertura")

cards = [
    {
        "ax_idx":  1,
        "valor":   f"{pct_ok:.1f}%",
        "status":  lbl_cob,
        "cor":     cor_cob,
        "titulo":  "Cobertura IBGE",
        "detalhe": f"{c_match} de {total} lojas identificadas",
    },
    {
        "ax_idx":  2,
        "valor":   f"{pdf_com.iloc[0]['populacao_int']:,}" if not pdf_com.empty else "—",
        "status":  "Maior mercado",
        "cor":     COR_AZUL,
        "titulo":  "Município de Referência",
        "detalhe": (
            f"{pdf_com.iloc[0]['cidade_loja']}/{pdf_com.iloc[0]['estado_tratado']} "
            f"({pdf_com.iloc[0]['nome_loja']})"
            if not pdf_com.empty else "Sem dados IBGE"
        ),
    },
]

for card in cards:
    ax = fig.add_subplot(1, 2, card["ax_idx"])
    ax.set_facecolor("white")
    ax.axis("off")

    # Linha colorida no topo
    ax.plot([0.08, 0.92], [0.97, 0.97], color=card["cor"],
            linewidth=5, solid_capstyle="butt",
            transform=ax.transAxes, clip_on=False)

    # Valor principal
    ax.text(0.5, 0.68, card["valor"],
            ha="center", va="center",
            fontsize=36, fontweight="bold", color=card["cor"],
            transform=ax.transAxes)

    # Status
    ax.text(0.5, 0.50, card["status"],
            ha="center", va="center",
            fontsize=9, fontweight="bold", color=card["cor"],
            transform=ax.transAxes)

    # Separador
    ax.plot([0.10, 0.90], [0.42, 0.42], color="#E0E0E0",
            linewidth=0.8, transform=ax.transAxes, clip_on=False)

    # Título
    ax.text(0.5, 0.30, card["titulo"],
            ha="center", va="center",
            fontsize=9, color="#424242", fontweight="bold",
            transform=ax.transAxes)

    # Detalhe
    ax.text(0.5, 0.12, card["detalhe"],
            ha="center", va="center",
            fontsize=7.5, color="#757575",
            transform=ax.transAxes, wrap=True)

    # Borda sutil
    for side in ["top", "bottom", "left", "right"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#E0E0E0")
        ax.spines[side].set_linewidth(0.8)

fig.suptitle("Contexto Demográfico — IBGE 2022",
             fontsize=12, fontweight="bold", color="#212121", y=1.04)
plt.tight_layout(pad=1.2)
plt.show()

if not pdf_com.empty:
    pdf_com_sorted = pdf_com.sort_values("populacao_int", ascending=True)
    n = len(pdf_com_sorted)
    x_max = pdf_com_sorted["populacao_int"].max()

    # ── Plotly — barras horizontais minimalistas ──────────────────────────────
    # Cor única sólida (sem gradiente) com texto dentro — legível e limpo
    fig = go.Figure(go.Bar(
        y=pdf_com_sorted["nome_loja"],
        x=pdf_com_sorted["populacao_int"],
        orientation="h",
        marker_color=COR_AZUL,
        marker_line=dict(color="white", width=0.5),
        text=pdf_com_sorted["populacao_int"].apply(lambda x: f"{x:,}"),
        textposition="inside",
        textfont=dict(color="white", size=10),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Cidade: <b>%{customdata}</b><br>"
            "Habitantes: <b>%{x:,}</b><extra></extra>"
        ),
        customdata=pdf_com_sorted.apply(
            lambda r: f"{r['cidade_loja']}/{r['estado_tratado']}", axis=1
        ),
        width=0.6,
    ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="População do Município por Loja (IBGE 2022)",
                   font=dict(size=14, color="#212121")),
        xaxis=dict(
            title="Habitantes",
            tickformat=",",
            range=[0, x_max * 1.05],
            gridcolor="#F5F5F5",
            showgrid=True,
            zeroline=False,
            tickfont=dict(color="#424242", size=9),
        ),
        yaxis=dict(
            title="",
            tickfont=dict(color="#212121", size=10),
            showgrid=False,
        ),
        height=max(360, n * 38),
        margin=dict(t=65, b=55, l=160, r=40),
    )
    fig.show()

    # ── Matplotlib (estático) ─────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, max(3.5, n * 0.45)),
                           facecolor="white")
    ax.set_facecolor("white")
    bars = ax.barh(pdf_com_sorted["nome_loja"],
                   pdf_com_sorted["populacao_int"],
                   color=COR_AZUL, edgecolor="white", linewidth=0.5,
                   height=0.6)
    # Labels dentro das barras (branco sobre azul — contraste garantido)
    for bar in bars:
        w = bar.get_width()
        if w > x_max * 0.15:  # só coloca texto se a barra for larga o suficiente
            ax.text(w * 0.5, bar.get_y() + bar.get_height() / 2,
                    f"{int(w):,}", ha="center", va="center",
                    fontsize=8, color="white", fontweight="bold")
    ax.invert_yaxis()
    ax.set_title("Lojas por Tamanho do Município — IBGE 2022",
                 fontsize=11, fontweight="bold", color="#212121", pad=12)
    ax.set_xlabel("Habitantes", color="#616161", fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.tick_params(colors="#424242", labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#E0E0E0")
    ax.spines["bottom"].set_color("#E0E0E0")
    ax.grid(axis="x", color="#F5F5F5", linewidth=0.8)
    plt.tight_layout()
    plt.show()

---
##  KPI 8 — Ticket Médio por Semestre
*Valor médio gasto por transação. Indica o "tamanho do carrinho" por loja e período.*

In [0]:
df_ticket = (
    df_gold
    .select("nome_loja", "ano", "semestre", "ticket_medio")
    .distinct()
    .filter(col("ticket_medio").isNotNull())
    .orderBy("nome_loja", "ano", "semestre")
)
pdf_ticket = df_ticket.toPandas()

if not pdf_ticket.empty:
    pdf_ticket["periodo"] = (
        pdf_ticket["ano"].astype(str) + " S" +
        pdf_ticket["semestre"].astype(str)
    )
    media_rede = pdf_ticket["ticket_medio"].mean()
    top_t = pdf_ticket.loc[pdf_ticket["ticket_medio"].idxmax()]
    bot_t = pdf_ticket.loc[pdf_ticket["ticket_medio"].idxmin()]
    print(f"🎯 Média da rede: {formatar_reais(media_rede)}")
    print(f"   🥇 Maior ticket: {top_t['nome_loja']} em {top_t['periodo']} — {formatar_reais(top_t['ticket_medio'])}")
    print(f"   🔻 Menor ticket: {bot_t['nome_loja']} em {bot_t['periodo']} — {formatar_reais(bot_t['ticket_medio'])}")

    # ── Plotly — barras agrupadas (cor única, texto dentro) ──────────────────
    periodos  = sorted(pdf_ticket["periodo"].unique())
    n_periodos = len(periodos)
    cores_per = PALETA_DISCRETA[:n_periodos]

    fig = go.Figure()
    for i, periodo in enumerate(periodos):
        df_p = pdf_ticket[pdf_ticket["periodo"] == periodo].sort_values("nome_loja")
        fig.add_trace(go.Bar(
            name=periodo,
            x=df_p["nome_loja"],
            y=df_p["ticket_medio"],
            marker_color=cores_per[i],
            marker_line=dict(color="white", width=0.5),
            text=df_p["ticket_medio"].apply(lambda x: f"R$ {x:,.0f}"),
            textposition="inside",
            textfont=dict(color="white", size=9),
            hovertemplate=(
                f"<b>%{{x}}</b> — {periodo}<br>"
                "Ticket: <b>R$ %{y:,.2f}</b><extra></extra>"
            ),
            width=0.7 / n_periodos,
        ))

    # Linha de média da rede
    fig.add_hline(
        y=media_rede,
        line_dash="dot",
        line_color=COR_CINZA,
        line_width=2,
        annotation_text=f"Média rede: {formatar_reais(media_rede)}",
        annotation_font_color=COR_CINZA,
        annotation_font_size=10,
        annotation_position="top right",
    )
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text="Ticket Médio por Loja e Semestre",
                   font=dict(size=14, color="#212121")),
        barmode="group",
        xaxis=dict(
            title="Loja",
            # tickangle com ha automático evita sobreposição de nomes longos
            tickangle=-35,
            tickfont=dict(color="#212121", size=9),
        ),
        yaxis=dict(
            title="Ticket Médio (R$)",
            tickprefix="R$ ",
            tickformat=",.2f",
            gridcolor="#E0E0E0",
            tickfont=dict(color="#424242"),
        ),
        height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02,
                    xanchor="left", x=0, font=dict(size=10)),
        margin=dict(t=100, b=100, l=100, r=120),
    )
    fig.show()

    # ── Matplotlib (estático) ────────────────────────────────────────────────
    pdf_piv = pdf_ticket.pivot_table(
        index="nome_loja", columns="periodo",
        values="ticket_medio", aggfunc="mean"
    ).fillna(0)
    n_col = len(pdf_piv.columns)
    fig, ax = plt.subplots(figsize=(max(10, n_col * 2.5), max(4, len(pdf_piv) * 0.6)))
    pdf_piv.plot(kind="bar", ax=ax, color=PALETA_DISCRETA[:n_col],
                 edgecolor="white", linewidth=0.5)
    ax.axhline(media_rede, color=COR_CINZA, linestyle="--",
               linewidth=1.5, label=f"Média rede: {formatar_reais(media_rede)}")
    ax.set_title("Ticket Médio por Loja e Semestre", fontsize=12,
                 fontweight="bold", color="#212121")
    ax.set_ylabel("Ticket Médio (R$)", color="#424242")
    ax.set_xlabel("")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x:,.2f}"))
    # ha="right" evita sobreposição de labels nos ticks do eixo X
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
    ax.legend(title="Período", fontsize=8, title_fontsize=9, framealpha=0.9)
    ax.tick_params(colors="#424242")
    plt.tight_layout()
    plt.show()

---
##  Painel de Alertas
*Lojas que requerem atenção da gestão com base nos indicadores acima.*

In [0]:
print("=" * 66)
print("   🚨  PAINEL DE ALERTAS — LOJAS FÍSICAS")
print("=" * 66)

alertas = []

# Alerta 1: YoY negativo
if not pdf_yoy.empty and "pdf_yoy_ref" in dir():
    lojas_neg = pdf_yoy_ref[pdf_yoy_ref["crescimento_yoy_pct"] < 0]
    for _, row in lojas_neg.iterrows():
        alertas.append(("🔴 CRÍTICO", row["nome_loja"], "Crescimento YoY",
                        f"Queda de {abs(row['crescimento_yoy_pct']):.1f}% vs ano anterior"))

# Alerta 2: Sem match IBGE
if not pdf_pop.empty:
    for _, row in pdf_pop[pdf_pop["flag_sem_match_ibge"] == True].iterrows():
        alertas.append(("🟡 ATENÇÃO", row["nome_loja"], "Dado IBGE ausente",
                        f"Cidade '{row['cidade_loja']}' não encontrada no cadastro IBGE"))

# Alerta 3: Ticket abaixo da média
if not pdf_ticket.empty:
    ticket_rec = pdf_ticket[pdf_ticket["ano"] == ano_ref]
    ticket_med_loja = ticket_rec.groupby("nome_loja")["ticket_medio"].mean()
    for loja, t in ticket_med_loja.items():
        if t < media_rede * 0.80:
            alertas.append(("🟡 ATENÇÃO", loja, "Ticket médio baixo",
                            f"{formatar_reais(t)} (>20% abaixo da média da rede: {formatar_reais(media_rede)})"))

if alertas:
    for nivel, loja, indicador, detalhe in alertas:
        print(f"\n  {nivel}")
        print(f"  {'Loja':<12}: {loja}")
        print(f"  {'Indicador':<12}: {indicador}")
        print(f"  {'Detalhe':<12}: {detalhe}")
        print(f"  {'-'*60}")
else:
    print("\n  ✅ Nenhum alerta identificado.")
    print("     Todos os indicadores dentro do esperado.")

print(f"\n  Total de alertas: {len(alertas)}")
print("=" * 66)

---
## 📋 Próximos Passos

| # | Ação | Responsável | Prazo |
|---|------|-------------|-------|
| 1 | Validar alertas acima com os gerentes regionais | Gestão Comercial | Imediato |
| 2 | Investigar lojas com YoY negativo | Gerente de Loja | 15 dias |
| 3 | Corrigir cidades sem match IBGE no cadastro | Dados / Operações | 30 dias |
| 4 | Dashboard Looker conectado a `squad3.gold_physical_lojas` | Squad 3 | Próxima sprint |

---
> **Fonte dos dados:** Azure Data Lake (Raw → Bronze → Silver → Gold)
> **Tabela SQL Server:** `squad3.gold_physical_lojas`
> **Atualização:** execução manual do pipeline Squad 3